In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.default;
CREATE VOLUME IF NOT EXISTS workspace.default.day17_lab;

CREATE TABLE IF NOT EXISTS workspace.default.day17_customer_bronze (
  CustomerId INT,
  CustomerName STRING,
  City STRING,
  Age INT,
  UpdatedAt TIMESTAMP,
  SourceFile STRING,
  IngestedAt TIMESTAMP,
  EventHash STRING
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.default.day17_customer_scd2 (
  CustomerId INT,
  CustomerName STRING,
  City STRING,
  Age INT,
  ValidFrom TIMESTAMP,
  ValidTo TIMESTAMP,
  IsCurrent BOOLEAN,
  EventHash STRING
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.default.day17_customer_late_events (
  CustomerId INT,
  CustomerName STRING,
  City STRING,
  Age INT,
  UpdatedAt TIMESTAMP,
  EventHash STRING,
  LateReason STRING,
  SourceFile STRING,
  RejectedAt TIMESTAMP
) USING DELTA;

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.window import Window

catalog_schema = "workspace.default"

incoming_path = "/Volumes/workspace/default/day17_lab/incoming_test"
checkpoint_path = "/Volumes/workspace/default/day17_lab/checkpoints/customer_scd2_04"

bronze_table = f"{catalog_schema}.day17_customer_bronze"
silver_table = f"{catalog_schema}.day17_customer_scd2"
late_table = f"{catalog_schema}.day17_customer_late_events"

customer_schema = StructType([
    StructField("CustomerId", IntegerType(), False),
    StructField("CustomerName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("UpdatedAt", TimestampType(), False)
])

In [0]:
raw_stream_df = (
    spark.readStream
    .format("csv")
    .schema(customer_schema)
    .option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .load(incoming_path)
)

In [0]:
hash_expression = F.sha2(
    F.concat_ws(
        "||",
        F.coalesce(F.col("CustomerName"), F.lit("∅")),
        F.coalesce(F.col("City"), F.lit("∅")),
        F.coalesce(F.col("Age").cast("string"), F.lit("∅"))
    ),
    256
)

prepared_stream_df = (
    raw_stream_df
    .withColumn("SourceFile", F.col("_metadata.file_path"))
    .withColumn("IngestedAt", F.current_timestamp())
    .withColumn("EventHash", hash_expression)
)


In [0]:
def process_customer_scd2(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    # Keep the most recent event for each customer within this micro-batch.
    # This prevents two incoming rows for one customer from producing
    # duplicate current records in a single batch.
    latest_per_customer_window = (
        Window.partitionBy("CustomerId")
        .orderBy(F.col("UpdatedAt").desc(), F.col("EventHash").desc())
    )

    deduped_batch_df = (
        batch_df
        .withColumn("row_number", F.row_number().over(latest_per_customer_window))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
    )

    # Bronze keeps the raw accepted file events for replay, audit, and debugging.
    bronze_to_write = deduped_batch_df.select(
        "CustomerId",
        "CustomerName",
        "City",
        "Age",
        "UpdatedAt",
        "SourceFile",
        "IngestedAt",
        "EventHash"
    )
    bronze_to_write.write.format("delta").mode("append").saveAsTable(bronze_table)

    current_silver_df = (
        spark.table(silver_table)
        .filter(F.col("IsCurrent") == True)
        .select(
            F.col("CustomerId").alias("CurrentCustomerId"),
            F.col("ValidFrom").alias("CurrentValidFrom"),
            F.col("EventHash").alias("CurrentEventHash")
        )
    )

    compared_df = (
        deduped_batch_df.alias("s")
        .join(
            current_silver_df.alias("t"),
            F.col("s.CustomerId") == F.col("t.CurrentCustomerId"),
            "left"
        )
        .select(
            F.col("s.CustomerId"),
            F.col("s.CustomerName"),
            F.col("s.City"),
            F.col("s.Age"),
            F.col("s.UpdatedAt"),
            F.col("s.SourceFile"),
            F.col("s.EventHash"),
            F.col("t.CurrentCustomerId"),
            F.col("t.CurrentValidFrom"),
            F.col("t.CurrentEventHash")
        )
    )

    # A changed event whose effective time is older than the current version
    # must not rewrite history accidentally. Preserve it in the late-events table.
    late_changed_df = (
        compared_df
        .filter(
            F.col("CurrentCustomerId").isNotNull()
            & (F.col("UpdatedAt") <= F.col("CurrentValidFrom"))
            & (F.col("EventHash") != F.col("CurrentEventHash"))
        )
        .select(
            "CustomerId",
            "CustomerName",
            "City",
            "Age",
            "UpdatedAt",
            "EventHash",
            F.lit("Changed event is older than or equal to the current SCD2 version").alias("LateReason"),
            "SourceFile",
            F.current_timestamp().alias("RejectedAt")
        )
    )

    if not late_changed_df.isEmpty():
        late_changed_df.write.format("delta").mode("append").saveAsTable(late_table)

    eligible_df = compared_df.filter(
        F.col("CurrentCustomerId").isNull()
        | (F.col("UpdatedAt") > F.col("CurrentValidFrom"))
    )

    # New customer: no current target row.
    # Changed customer: current target exists but tracked attributes differ.
    # Unchanged customer: hash matches, so it is intentionally excluded.
    new_or_changed_df = (
        eligible_df
        .filter(
            F.col("CurrentCustomerId").isNull()
            | (F.col("EventHash") != F.col("CurrentEventHash"))
        )
        .withColumn(
            "ChangeType",
            F.when(F.col("CurrentCustomerId").isNull(), F.lit("NEW"))
            .otherwise(F.lit("CHANGED"))
        )
        .select(
            "CustomerId",
            "CustomerName",
            "City",
            "Age",
            F.col("UpdatedAt").alias("ValidFrom"),
            "EventHash",
            "ChangeType"
        )
    )

    # For a changed customer, create two staging rows:
    # 1. CLOSE matches its current Silver row.
    # 2. INSERT has a null merge key, so it does not match and becomes a new version.
    close_stage_df = (
        new_or_changed_df
        .filter(F.col("ChangeType") == "CHANGED")
        .withColumn("MergeKey", F.col("CustomerId"))
        .withColumn("Action", F.lit("CLOSE"))
    )

    insert_stage_df = (
        new_or_changed_df
        .withColumn("MergeKey", F.lit(None).cast("int"))
        .withColumn("Action", F.lit("INSERT"))
    )

    staged_changes_df = close_stage_df.unionByName(insert_stage_df)

    if not staged_changes_df.isEmpty():
        silver_delta = DeltaTable.forName(spark, silver_table)

        (
            silver_delta.alias("t")
            .merge(
                staged_changes_df.alias("s"),
                "t.CustomerId = s.MergeKey AND t.IsCurrent = true"
            )
            .whenMatchedUpdate(
                condition="s.Action = 'CLOSE'",
                set={
                    "IsCurrent": "false",
                    "ValidTo": "s.ValidFrom"
                }
            )
            .whenNotMatchedInsert(
                condition="s.Action = 'INSERT'",
                values={
                    "CustomerId": "s.CustomerId",
                    "CustomerName": "s.CustomerName",
                    "City": "s.City",
                    "Age": "s.Age",
                    "ValidFrom": "s.ValidFrom",
                    "ValidTo": "CAST(NULL AS TIMESTAMP)",
                    "IsCurrent": "true",
                    "EventHash": "s.EventHash"
                }
            )
            .execute()
        )

In [0]:
query = (
    prepared_stream_df.writeStream
    .foreachBatch(process_customer_scd2)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
%sql
select * from day17_customer_bronze order by CustomerId;

CustomerId,CustomerName,City,Age,UpdatedAt,SourceFile,IngestedAt,EventHash
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z,dbfs:/Volumes/workspace/default/day17_lab/incoming_test/day17_batch_01.csv,2026-08-29T08:16:41.496Z,879526e89a7f5c6c8dcca0df4f6f7dfa35602c99545bfa568f864e8957babe7a
101,Arun,Chennai,31,2026-08-03T08:00:00.000Z,dbfs:/Volumes/workspace/default/day17_lab/incoming_test/day17_batch_03.csv,2026-08-29T08:29:29.147Z,d7f0a02bf6ec5731fad053075ae87b4600cd4af4ec48558b944fd42a86f1a53f
102,Kumar,Mysore,36,2026-08-01T08:00:00.000Z,dbfs:/Volumes/workspace/default/day17_lab/incoming_test/day17_batch_03.csv,2026-08-29T08:29:29.147Z,cfa6ab2b4b716086af81b1e82bd26a3cb65c373327ac0796b4423b45a1497946
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z,dbfs:/Volumes/workspace/default/day17_lab/incoming_test/day17_batch_01.csv,2026-08-29T08:16:41.496Z,5134a5bc1f2e47af4d86d3c595662b3c54e8cbaafd6dc6f702bf5abfe96e9371
102,Kumar,Bangalore,36,2026-08-02T10:00:00.000Z,dbfs:/Volumes/workspace/default/day17_lab/incoming_test/day17_batch_02.csv,2026-08-29T08:24:42.671Z,ff26aad79faeb7b8d31dad636643b671c8494927155d5e75c9c58ea30563a2a0
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z,dbfs:/Volumes/workspace/default/day17_lab/incoming_test/day17_batch_01.csv,2026-08-29T08:16:41.496Z,6b00729c73ee7053b526b2716842128153f600c3bf007bfa2193587b192e0fdf
103,Priya,Chennai,27,2026-08-02T10:05:00.000Z,dbfs:/Volumes/workspace/default/day17_lab/incoming_test/day17_batch_02.csv,2026-08-29T08:24:42.671Z,6b00729c73ee7053b526b2716842128153f600c3bf007bfa2193587b192e0fdf
104,Divya,Hyderabad,29,2026-08-02T10:11:00.000Z,dbfs:/Volumes/workspace/default/day17_lab/incoming_test/day17_batch_02.csv,2026-08-29T08:24:42.671Z,8cd77cb27260e28bdd724f4f3ac57d8d1f88a555db261c29755f6fae381502f4


In [0]:
%sql
SELECT
  CustomerId,
  CustomerName,
  City,
  Age,
  ValidFrom,
  ValidTo,
  IsCurrent
FROM workspace.default.day17_customer_scd2
ORDER BY CustomerId, ValidFrom;

CustomerId,CustomerName,City,Age,ValidFrom,ValidTo,IsCurrent
101,Arun,Chennai,30,2026-08-01T09:00:00.000Z,2026-08-03T08:00:00.000Z,false
101,Arun,Chennai,31,2026-08-03T08:00:00.000Z,null,true
102,Kumar,Bangalore,35,2026-08-01T09:05:00.000Z,2026-08-02T10:00:00.000Z,false
102,Kumar,Bangalore,36,2026-08-02T10:00:00.000Z,null,true
103,Priya,Chennai,27,2026-08-01T09:10:00.000Z,null,true
104,Divya,Hyderabad,29,2026-08-02T10:11:00.000Z,null,true


In [0]:
%sql
SELECT
  CustomerId,
  City,
  Age,
  UpdatedAt,
  LateReason,
  RejectedAt
FROM workspace.default.day17_customer_late_events
ORDER BY RejectedAt;

CustomerId,City,Age,UpdatedAt,LateReason,RejectedAt
102,Mysore,36,2026-08-01T08:00:00.000Z,Changed event is older than or equal to the current SCD2 version,2026-08-29T08:29:33.020Z


In [0]:
%sql
SELECT
  CustomerId,
  COUNT(*) AS CurrentRowCount
FROM workspace.default.day17_customer_scd2
WHERE IsCurrent = true
GROUP BY CustomerId
HAVING COUNT(*) <> 1;

CustomerId,CurrentRowCount
